# Maximal Marginal Relevance

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader('langchain_rag_dataset.txt')
raw_docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [6]:
retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={"k":3}
)

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model

prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context: 
{context}
                                      
Question: {input}                      
""")

llm = init_chat_model("groq:llama-3.1-8b-instant")

In [9]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context":retriever, "input":RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [10]:
query = 'How does langchain support agents and memory?'

response = rag_chain.invoke(query)
print(response)

Based on the provided context, LangChain supports agents and memory in the following ways:

1. **Agents**: LangChain allows Large Language Models (LLMs) to act as agents that decide which tool to call and in what order during a task. Agents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.

2. **Memory**: LangChain helps models retain previous interactions, making multi-turn conversations more coherent. It supports conversational memory using ConversationBufferMemory and summarization memory with ConversationSummaryMemory.


In [11]:
response

'Based on the provided context, LangChain supports agents and memory in the following ways:\n\n1. **Agents**: LangChain allows Large Language Models (LLMs) to act as agents that decide which tool to call and in what order during a task. Agents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.\n\n2. **Memory**: LangChain helps models retain previous interactions, making multi-turn conversations more coherent. It supports conversational memory using ConversationBufferMemory and summarization memory with ConversationSummaryMemory.'

In [13]:
docs = retriever.invoke(query)

for doc in docs:
    print(doc)
    print()

page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.
Agents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.' metadata={'source': 'langchain_rag_dataset.txt'}

page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.
RAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.' metadata={'source': 'langchain_rag_dataset.txt'}

page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.
LangChain supports conversational memory using ConversationBufferMemory and summarization memory with ConversationSummaryMemory.' metadata={'source': 'langchain_rag_dataset.txt'}

